In [17]:
### Install Dependencies
!pip install sagemaker langchain langchain-community boto3 transformers

In [18]:
import sagemaker
from sagemaker. huggingface import HuggingFaceModel
import boto3

# Configuration
role = sagemaker.get_execution_role()
region = "us-east-1"

# CORRECT Model configuration for FLAN-T5
hub = {
    'HF_MODEL_ID': 'google/flan-t5-base',
    'HF_TASK':  'text2text-generation',  
}

# Create HuggingFace Model
huggingface_model = HuggingFaceModel(
    model_data=None,
    role=role,
    transformers_version="4.26.0",  
    pytorch_version="1.13.1",       
    py_version="py39",              
    env=hub,
)

# Deploy to SageMaker endpoint
from datetime import datetime
timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
endpoint_name = f"flan-t5-base-{timestamp}"  
print(f"Deploying endpoint: {endpoint_name}")

predictor = huggingface_model. deploy(
    initial_instance_count=1,
    instance_type="ml.m5.xlarge",  
    endpoint_name=endpoint_name
)

print(f"Endpoint deployed: {predictor.endpoint_name}")


sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml
Deploying endpoint: flan-t5-base-20251226-154641
-----!Endpoint deployed: flan-t5-base-20251226-154641


In [21]:

# Test the endpoint
test_result = predictor.predict({
    "inputs": "What is machine learning?",
    "parameters": {
        "temperature": 0.7
    }
})

print(f"\nTest result: {test_result}")


Test result: [{'generated_text': 'Machine learning is a computer science technique that uses computer algorithms to learn information about people and places'}]


In [22]:
from langchain_community.llms. sagemaker_endpoint import (
    LLMContentHandler, 
    SagemakerEndpoint
)
from typing import Dict
import json

class FlanT5ContentHandler(LLMContentHandler):
    """Content handler for FLAN-T5 on SageMaker"""
    
    content_type = "application/json"
    accepts = "application/json"

    def transform_input(self, prompt: str, model_kwargs: Dict) -> bytes:
        """Format input for the SageMaker endpoint"""
        input_data = {
            "inputs": prompt,
            "parameters":  model_kwargs
        }
        return json.dumps(input_data).encode('utf-8')

    def transform_output(self, output: bytes) -> str:
        """Parse output from the SageMaker endpoint"""
        response = json.loads(output.read().decode("utf-8"))
        
        # Handle HuggingFace response format
        if isinstance(response, list) and len(response) > 0:
            return response[0].get('generated_text', '')
        return response.get('generated_text', str(response))

# Initialize the LLM
llm = SagemakerEndpoint(
    endpoint_name="flan-t5-base-20251226-154641",  # Replace with your endpoint
    region_name="us-east-1",
    content_handler=FlanT5ContentHandler()
)

# Test it
response = llm.invoke("What is machine learning?")
print(response)

Machine learning is a computer science technique that uses computer algorithms to learn information about people and places


In [25]:
# Tools
def calculator(expr: str) -> str:
    """Calculate math"""
    try:
        expr = expr.replace(" ", "").replace("=", "")
        if all(c in "0123456789+-*/()" for c in expr):
            return str(eval(expr))
    except:
        return "Error"

def get_aws_info(svc: str) -> str:
    """Get AWS info"""
    info = {
        "sagemaker": "Amazon SageMaker is a fully managed machine learning platform for building, training, and deploying ML models",
        "lambda": "AWS Lambda is a serverless compute service that runs code in response to events",
        "s3": "Amazon S3 is an object storage service for any amount of data",
    }
    return info.get(svc. lower().strip(), f"No information for '{svc}'")

tools_dict = {
    "Calculator": {"func": calculator, "desc": "Calculate math expressions"},
    "AWSInfo":  {"func": get_aws_info, "desc": "Get AWS service information"},
}

# Improved Agent with Direct Tool Invocation
def run_agent_smart(question: str):
    """Smart agent that detects when to use tools directly"""
    
    question_lower = question.lower()
   
    if any(word in question_lower for word in ["calculate", "what is", "multiply", "divide", "plus", "minus", "*", "/", "+"]):
        import re
        # Look for number patterns
        math_patterns = [
            r'(\d+)\s*\*\s*(\d+)',  # 25 * 4
            r'(\d+)\s*/\s*(\d+)',   # 100 / 5
            r'(\d+)\s*\+\s*(\d+)',  # 50 + 50
            r'(\d+)\s*-\s*(\d+)',   # 100 - 50
        ]
        
        for pattern in math_patterns:
            match = re. search(pattern, question)
            if match:
                expr = match.group(0)
                result = calculator(expr)
                return f"The answer is {result}"
    
    # Direct AWS info detection
    if any(word in question_lower for word in ["tell me about", "what is", "explain", "describe"]):
        for service in ["sagemaker", "lambda", "s3"]: 
            if service in question_lower:
                return get_aws_info(service)
    
    # If no direct match, use LLM
    prompt = f"""Answer this question directly and concisely:  {question}

Answer:"""
    
    response = llm.invoke(prompt).strip()
    return response

# Test
print("=" * 70)
print("Smart Agent with Direct Tool Detection")
print("=" * 70)

test_questions = [
    "What is 25 * 4? ",
    "Calculate 100 / 5",
    "Tell me about AWS SageMaker",
    "What is 50 + 30?",
    "Explain AWS Lambda",
]

for q in test_questions:
    print(f"\n{'=' * 70}")
    print(f"Q: {q}")
    print('=' * 70)
    answer = run_agent_smart(q)
    print(f"Answer: {answer}\n")

Smart Agent with Direct Tool Detection

Q: What is 25 * 4? 
Answer: The answer is 100


Q: Calculate 100 / 5
Answer: The answer is 20.0


Q: Tell me about AWS SageMaker
Answer: Amazon SageMaker is a fully managed machine learning platform for building, training, and deploying ML models


Q: What is 50 + 30?
Answer: The answer is 80


Q: Explain AWS Lambda
Answer: AWS Lambda is a serverless compute service that runs code in response to events



In [33]:
import re

def simple_agent(question: str, verbose: bool = True) -> str:
    """
    A rule-based agent that routes questions to appropriate tools
    
    Strategy:
    1. Check for math patterns → Use Calculator
    2. Check for AWS keywords → Use AWSInfo
    3. Otherwise → Use LLM directly
    """
    
    if verbose:
        print(f"\n{'='*60}")
        print(f"Question: {question}")
        print('='*60)
    
    question_lower = question.lower()
    
    # STRATEGY 1: Detect mathematical expressions
    math_keywords = ["calculate", "what is", "multiply", "divide", "*", "/", "+", "-"]
    if any(keyword in question_lower for keyword in math_keywords):
        
        # Regex patterns for different math operations
        math_patterns = [
            r'(\d+)\s*\*\s*(\d+)',      # Multiplication:  25 * 4
            r'(\d+)\s*/\s*(\d+)',        # Division: 100 / 5
            r'(\d+)\s*\+\s*(\d+)',       # Addition: 50 + 30
            r'(\d+)\s*-\s*(\d+)',        # Subtraction: 100 - 25
        ]
        
        for pattern in math_patterns:
            match = re.search(pattern, question)
            if match:
                expression = match.group(0)
                
                if verbose:
                    print(f"Tool Selected: Calculator")
                    print(f"Expression: {expression}")
                
                result = calculator(expression)
                
                if verbose:
                    print(f"Result:  {result}")
                
                return f"The answer is {result}"
    
    # STRATEGY 2: Detect AWS service questions
    aws_keywords = ["tell me about", "what is", "explain", "describe"]
    if any(keyword in question_lower for keyword in aws_keywords):
        
        aws_services = ["sagemaker", "lambda", "s3", "bedrock", "ec2", "dynamodb"]
        for service in aws_services:
            if service in question_lower:
                
                if verbose:
                    print(f"Tool Selected: AWSInfo")
                    print(f"Service: {service}")
                
                result = get_aws_info(service)
                
                if verbose:
                    print(f"Result: {result[: 100]}...")
                
                return result
    
    # STRATEGY 3: Use LLM for general questions
    if verbose:
        print(f"Tool Selected: LLM (no specific tool matched)")
    
    prompt = f"""Answer this question directly and concisely:

Question: {question}

Answer: """
    
    response = llm.invoke(prompt).strip()
    
    if verbose: 
        print(f"Result: {response}")
    
    return response

In [35]:
# Test different types of questions
test_questions = [
    "What is 25 * 4?",                    # Should use Calculator
    "Explain AWS Lambda",                  # Should use AWSInfo
    "What is the capital of France?",      # Should use LLM
]

print("="*60)
print("AGENT TESTING")
print("="*60)

for question in test_questions:
    answer = simple_agent(question, verbose=True)
    print(f"\nFinal Answer: {answer}\n")

AGENT TESTING

Question: What is 25 * 4?
Tool Selected: Calculator
Expression: 25 * 4
Result:  100

Final Answer: The answer is 100


Question: Explain AWS Lambda
Tool Selected: AWSInfo
Service: lambda
Result: AWS Lambda is a serverless compute service that runs code in response to events...

Final Answer: AWS Lambda is a serverless compute service that runs code in response to events


Question: What is the capital of France?
Tool Selected: LLM (no specific tool matched)
Result: london

Final Answer: london



In [36]:

# ============================================
# Conversation Memory (In-Memory)
# ============================================

class ConversationMemory:
    """Simple in-memory conversation tracker"""
    
    def __init__(self, max_history: int = 20):
        self.history:  List[Dict] = []
        self.max_history = max_history
        self.session_id = datetime.now().strftime("%Y%m%d-%H%M%S")
        print(f"✓ Session started: {self.session_id}")
    
    def add_interaction(self, question: str, answer: str, tool_used: str = None):
        """Add Q&A to history"""
        self.history. append({
            "timestamp": datetime.now().isoformat(),
            "question": question,
            "answer": answer,
            "tool_used": tool_used
        })
        
        # Keep only last N
        if len(self.history) > self.max_history:
            self.history = self.history[-self.max_history:]
    
    def get_context(self, num_turns: int = 3) -> str:
        """Get recent conversation context"""
        if not self.history:
            return "No previous conversation"
        
        recent = self.history[-num_turns:]
        lines = []
        
        for i, conv in enumerate(recent, 1):
            lines.append(f"Turn {i}:")
            lines.append(f"  User: {conv['question']}")
            lines.append(f"  Assistant: {conv['answer']}")
            if conv['tool_used']:
                lines. append(f"  [Tool: {conv['tool_used']}]")
        
        return "\n".join(lines)
    
    def get_summary(self) -> str:
        """Get conversation summary"""
        if not self.history:
            return "No conversation yet"
        
        tool_usage = {}
        for conv in self.history:
            tool = conv.get('tool_used', 'LLM')
            tool_usage[tool] = tool_usage. get(tool, 0) + 1
        
        summary = f"Session:  {self.session_id}\n"
        summary += f"Total turns: {len(self.history)}\n"
        summary += f"Tool usage: {json.dumps(tool_usage, indent=2)}"
        return summary
    
    def save_to_file(self, filename: str = None):
        """Save conversation to JSON file"""
        if filename is None:
            filename = f"conversation_{self.session_id}.json"
        
        data = {
            "session_id": self.session_id,
            "total_turns": len(self.history),
            "history": self. history
        }
        
        with open(filename, 'w') as f:
            json.dump(data, f, indent=2)
        
        print(f"Saved to {filename}")
        return filename
    
    def export_to_markdown(self, filename: str = None):
        """Export conversation to Markdown"""
        if filename is None:
            filename = f"conversation_{self.session_id}.md"
        
        with open(filename, 'w') as f:
            f.write(f"# Conversation Log\n\n")
            f.write(f"**Session ID:** {self.session_id}\n")
            f.write(f"**Total Turns:** {len(self.history)}\n\n")
            f.write(f"---\n\n")
            
            for i, conv in enumerate(self.history, 1):
                f.write(f"## Turn {i}\n\n")
                f.write(f"**Time:** {conv['timestamp']}\n\n")
                f.write(f"**User:** {conv['question']}\n\n")
                f.write(f"**Assistant:** {conv['answer']}\n\n")
                if conv['tool_used']:
                    f.write(f"*Tool used: {conv['tool_used']}*\n\n")
                f.write(f"---\n\n")
        
        print(f"Exported to {filename}")
        return filename

In [37]:
# ============================================
# Agent with Memory
# ============================================

class AgentWithMemory:
    """Agent that remembers conversation history"""
    
    def __init__(self, llm, memory:  ConversationMemory):
        self.llm = llm
        self.memory = memory
    
    def run(self, question: str, verbose: bool = True) -> str:
        """Process question with conversation context"""
        
        if verbose:
            print(f"\n{'='*70}")
            print(f"Question: {question}")
            print('='*70)
        
        question_lower = question.lower().strip()
        tool_used = None
        
        # Check for follow-up references
        follow_up_keywords = ["that", "it", "previous", "last", "earlier", "before", "first"]
        is_follow_up = any(word in question_lower for word in follow_up_keywords)
        
        if is_follow_up and self.memory.history:
            if verbose:
                print(f"Detected follow-up question")
                print(f"Context:\n{self.memory.get_context(num_turns=2)}\n")
        
        # Strategy 1: Math
        math_pattern = re.search(r'(\d+)\s*([*/+-])\s*(\d+)', question)
        if math_pattern: 
            expr = f"{math_pattern.group(1)}{math_pattern.group(2)}{math_pattern.group(3)}"
            result = calculator(expr)
            tool_used = "Calculator"
            
            if verbose: 
                print(f"Tool: Calculator")
                print(f"Input: {expr}")
                print(f"Output: {result}")
            
            answer = f"The answer is {result}"
            self.memory.add_interaction(question, answer, tool_used)
            return answer
        
        # Strategy 2: AWS Info
        for service in ["sagemaker", "lambda", "s3", "bedrock"]:
            if service in question_lower:
                result = get_aws_info(service)
                tool_used = "AWSInfo"
                
                if verbose:
                    print(f"Tool: AWSInfo")
                    print(f"Input: {service}")
                    print(f"Output: {result[: 80]}...")
                
                answer = result
                self.memory.add_interaction(question, answer, tool_used)
                return answer
        
        # Strategy 3: Memory-based questions
        if is_follow_up:
            if "first" in question_lower and self.memory.history:
                first_q = self.memory.history[0]['question']
                answer = f"Your first question was: '{first_q}'"
                self.memory.add_interaction(question, answer, "Memory")
                return answer
            
            if "last" in question_lower and len(self.memory.history) >= 2:
                last_q = self.memory.history[-2]['question']
                answer = f"Your last question was: '{last_q}'"
                self. memory.add_interaction(question, answer, "Memory")
                return answer
        
        # Strategy 4: LLM with context
        if verbose:
            print(f"Using LLM with context")
        
        context = self.memory. get_context(num_turns=3)
        prompt = f"""Previous conversation:
{context}

Current question:  {question}

Answer concisely.  If the question refers to the previous conversation, use that context. 

Answer: """
        
        answer = self.llm.invoke(prompt).strip()
        self.memory.add_interaction(question, answer, None)
        
        return answer

print("Agent with memory ready\n")


Agent with memory ready



In [38]:

# ============================================
# Demo:  Conversation with Memory
# ============================================

print("="*70)
print("DEMO: Agent with Conversation Memory")
print("="*70)

# Create memory and agent
memory = ConversationMemory(max_history=20)
agent = AgentWithMemory(llm, memory)

# Test conversation with follow-ups
conversation = [
    "What is 25 * 4?",
    "What about 100 / 5?",
    "Tell me about AWS SageMaker",
    "What is AWS Lambda?",
    "Calculate 50 + 30",
    "What was my first question?",  # Tests memory recall
    "And what was my last calculation? ",  # Tests context
]

for i, question in enumerate(conversation, 1):
    print(f"\n{'#'*70}")
    print(f"Turn {i}/{len(conversation)}")
    print(f"{'#'*70}")
    
    answer = agent.run(question, verbose=True)
    print(f"\nANSWER: {answer}")

# Show summary
print(f"\n\n{'='*70}")
print("CONVERSATION SUMMARY")
print('='*70)
print(memory.get_summary())

# Show full history
print(f"\n{'='*70}")
print("FULL CONVERSATION HISTORY")
print('='*70)
print(memory.get_context(num_turns=len(memory.history)))

# Save conversation
json_file = memory.save_to_file()
md_file = memory.export_to_markdown()

print(f"\n{'='*70}")
print("FILES SAVED")
print('='*70)
print(f"  • JSON: {json_file}")
print(f"  • Markdown:  {md_file}")

DEMO: Agent with Conversation Memory
✓ Session started: 20251226-160837

######################################################################
Turn 1/7
######################################################################

Question: What is 25 * 4?
Tool: Calculator
Input: 25*4
Output: 100

ANSWER: The answer is 100

######################################################################
Turn 2/7
######################################################################

Question: What about 100 / 5?
Tool: Calculator
Input: 100/5
Output: 20.0

ANSWER: The answer is 20.0

######################################################################
Turn 3/7
######################################################################

Question: Tell me about AWS SageMaker
Tool: AWSInfo
Input: sagemaker
Output: Amazon SageMaker is a fully managed machine learning platform for building, trai...

ANSWER: Amazon SageMaker is a fully managed machine learning platform for building, training, and deploying ML m